<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_02_data_and_sampling_distributions/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 2 folder
%cd chapter_02_data_and_sampling_distributions

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 152 (delta 90), reused 71 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 530.38 KiB | 6.10 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_02_data_and_sampling_distributions


I'll combine both responses into a comprehensive Chapter 2 exercises notebook that includes all content from both sources without mentioning their origins.

# Chapter 02 Exercises: Data and Sampling Distributions

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 2: Data and Sampling Distributions

## Goals

In this notebook, I will practise:
- Population vs sample
- Random sampling and sampling bias
- Regression to the mean
- Sampling distributions
- Central Limit Theorem (CLT)
- Standard error
- Bootstrap
- Confidence intervals
- Normal distributions and QQ-plots
- Distribution identification and fitting

---

## 📋 Instructions

- Attempt each exercise before viewing the solution
- Use `utils/notebook_setup.py` for consistent imports when available
- Save your work frequently and commit progress to GitHub
- Solutions are provided in collapsed sections—expand only after attempting the problem

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from scipy.stats import sem, skew, kurtosis, norm, t, binom, poisson, expon, uniform, gamma, lognorm
    from scipy.stats import shapiro, kstest, proportions_ztest
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Helper function for bootstrap
def bootstrap_statistic(data, statistic_func, R=1000, random_state=None):
    """Generic bootstrap function."""
    if random_state is not None:
        np.random.seed(random_state)
    
    bootstrap_stats = []
    n = len(data)
    
    for _ in range(R):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats.append(statistic_func(resample))
    
    return np.array(bootstrap_stats)

print("Exercise notebook ready")
```
</details>


---

## Load Dataset

<details>
<summary>Click to view solution</summary>

```python
# Load state dataset
data_path = '../datasets/raw/state.csv'
if os.path.exists(data_path):
    state = pd.read_csv(data_path)
else:
    url = 'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/state.csv'
    state = pd.read_csv(url)

state.head()
```
</details>


---

# Exercise 1: Population vs Sample

## Task

Create a sample of 10 states. Compare population mean and sample mean.

### Questions

1. Are they identical?
2. Why are they different?
3. Can a sample still be useful?

<details>
<summary>Click to view solution</summary>

```python
sample = state.sample(n=10, random_state=42)

population_mean = state["Population"].mean()
sample_mean = sample["Population"].mean()

print(f"Population Mean: {population_mean:,.2f}")
print(f"Sample Mean: {sample_mean:,.2f}")
print(f"Difference: {abs(population_mean - sample_mean):,.2f}")
```
</details>



---

# Exercise 2: Random Sampling

## Task

Take 5 different random samples. Compare their means.

### Question

Do all samples produce identical results?

<details>
<summary>Click to view solution</summary>

```python
for i in range(5):
    temp_sample = state.sample(n=10)
    print(f"Sample {i+1} Mean: {temp_sample['Population'].mean():,.2f}")

print(f"\nPopulation Mean: {state['Population'].mean():,.2f}")
```
</details>

## Reflection

Questions:
1. Why are sample means different?
2. Does this mean sampling is unreliable?



---

# Exercise 3: Sampling Bias

## Task

Create a biased sample using largest states only. Compare with random sample.

<details>
<summary>Click to view solution</summary>

```python
random_sample = state.sample(n=10, random_state=42)
biased_sample = state.nlargest(10, "Population")

print("Population Mean:", f"{state['Population'].mean():,.2f}")
print("Random Sample Mean:", f"{random_sample['Population'].mean():,.2f}")
print("Biased Sample Mean:", f"{biased_sample['Population'].mean():,.2f}")

# Visualization
comparison = pd.DataFrame({
    "Population": state["Population"],
    "Biased Sample": biased_sample["Population"]
})

comparison.plot(kind="box", figsize=(8, 5))
plt.title("Population vs Biased Sample")
plt.show()
```
</details>

## Questions

1. Which sample is closer to population?
2. Why is the biased sample misleading?
3. Can large datasets still be biased?



---

# Exercise 4: Simulating Sampling Bias (Literary Digest Poll)

## Task

Create a synthetic population where 70% prefer Option A and 30% prefer Option B. Then:
1. Draw a truly random sample of n=200 and compute the proportion preferring A
2. Draw a biased sample where high-income individuals (who prefer B at 60%) are overrepresented
3. Compare both sample estimates to the true population proportion

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Create synthetic population
n_pop = 100000
income = np.random.choice(['low', 'high'], size=n_pop, p=[0.8, 0.2])
preference = np.where(
    income == 'low',
    np.random.choice(['A', 'B'], size=n_pop, p=[0.85, 0.15]),
    np.random.choice(['A', 'B'], size=n_pop, p=[0.40, 0.60])
)

true_prop_A = np.mean(preference == 'A')
print(f"True population proportion preferring A: {true_prop_A:.2%}")

# Random sample (n=200)
random_idx = np.random.choice(n_pop, size=200, replace=False)
random_sample_pref = preference[random_idx]
random_prop_A = np.mean(random_sample_pref == 'A')

# Biased sample (overrepresent high-income)
low_income_idx = np.where(income == 'low')[0]
high_income_idx = np.where(income == 'high')[0]

biased_sample_idx = np.concatenate([
    np.random.choice(low_income_idx, size=100, replace=False),
    np.random.choice(high_income_idx, size=100, replace=False)
])
biased_sample_pref = preference[biased_sample_idx]
biased_prop_A = np.mean(biased_sample_pref == 'A')

# Results
print(f"\nRandom Sample (n=200):")
print(f"  Proportion preferring A: {random_prop_A:.2%}")
print(f"  Error: {abs(random_prop_A - true_prop_A):.2%}")

print(f"\nBiased Sample (n=200, 50% high-income):")
print(f"  Proportion preferring A: {biased_prop_A:.2%}")
print(f"  Error: {abs(biased_prop_A - true_prop_A):.2%}")

# Visualization
plt.figure(figsize=(8, 5))
plt.bar(['True', 'Random', 'Biased'],
        [true_prop_A, random_prop_A, biased_prop_A],
        color=['green', 'skyblue', 'salmon'], edgecolor='black')
plt.axhline(y=true_prop_A, color='green', linestyle='--', alpha=0.5)
plt.ylabel('Proportion Preferring A')
plt.title('Effect of Sampling Bias on Estimate')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInsight: Biased sampling can produce estimates far from truth, even with same n!")
```
</details>



---

# Exercise 5: Self-Selection Bias Simulation

## Task

Simulate session data with a `converted` column. Users who convert are more likely to complete a post-session survey. Simulate this by:
1. Creating a full dataset where conversion rate is 15%
2. Creating a "survey respondent" subset where converters are 3× more likely to respond
3. Compare conversion rates between full data and survey subset

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Simulate full session data
n_sessions = 5000
converted = np.random.binomial(1, 0.15, n_sessions)

# Simulate survey response bias
base_response_rate = 0.20
response_prob = np.where(converted == 1, base_response_rate * 3, base_response_rate)
responded = np.random.binomial(1, response_prob)

# Create DataFrame
df = pd.DataFrame({
    'session_id': range(n_sessions),
    'converted': converted,
    'survey_responded': responded
})

# Compare conversion rates
full_conversion_rate = df['converted'].mean()
survey_subset = df[df['survey_responded'] == 1]
survey_conversion_rate = survey_subset['converted'].mean()

print(f"Full Dataset (n={len(df)}):")
print(f"  Conversion Rate: {full_conversion_rate:.2%}")

print(f"\nSurvey Respondents (n={len(survey_subset)}):")
print(f"  Conversion Rate: {survey_conversion_rate:.2%}")
print(f"  Bias: {(survey_conversion_rate - full_conversion_rate):.2%} (overestimate!)")

# Response rate by conversion status
response_by_conv = df.groupby('converted')['survey_responded'].mean()
print(f"\nSurvey Response Rate by Conversion:")
print(f"  Non-converters: {response_by_conv[0]:.2%}")
print(f"  Converters: {response_by_conv[1]:.2%}")

# Visualization
plt.figure(figsize=(8, 5))
plt.bar(['Full Data', 'Survey Subset'],
        [full_conversion_rate, survey_conversion_rate],
        color=['skyblue', 'salmon'], edgecolor='black')
plt.axhline(y=full_conversion_rate, color='green', linestyle='--', alpha=0.5, label='True Rate')
plt.ylabel('Conversion Rate')
plt.title('Self-Selection Bias: Survey Respondents Overestimate Conversion')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nML Relevance: If you train a model only on survey respondents, it will over-predict conversion!")
```
</details>



---

# Exercise 6: Sampling Variability

## Task

Create 100 sample means using sample size 10. Visualise the distribution.

<details>
<summary>Click to view solution</summary>

```python
sample_means = []

for _ in range(100):
    s = state.sample(n=10, replace=True)
    sample_means.append(s["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(sample_means, bins=20, kde=True)
plt.title("Distribution of Sample Means")
plt.xlabel("Sample Mean")
plt.show()
```
</details>

## Reflection

Questions:
1. Why do means vary?
2. Why do they form a distribution?

---

# Exercise 7: Central Limit Theorem

## Task

Compare sampling distributions for n=5, n=30, n=50. Observe how sample size changes shape.

<details>
<summary>Click to view solution</summary>

```python
means_5 = []
means_30 = []
means_50 = []

for _ in range(1000):
    means_5.append(state.sample(5, replace=True)["Population"].mean())
    means_30.append(state.sample(30, replace=True)["Population"].mean())
    means_50.append(state.sample(50, replace=True)["Population"].mean())

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(means_5, kde=True, ax=ax[0])
ax[0].set_title("n = 5")

sns.histplot(means_30, kde=True, ax=ax[1])
ax[1].set_title("n = 30")

sns.histplot(means_50, kde=True, ax=ax[2])
ax[2].set_title("n = 50")

plt.show()
```
</details>

## Questions

1. Which looks most normal?
2. Why does larger sample size help?
3. Why is CLT important?


---

# Exercise 8: Visualizing CLT with Different Population Shapes

## Task

For each population distribution below, generate sampling distributions of the mean for n=5, 20, 50:
1. Uniform(0, 1)
2. Exponential(λ=1)
3. Binomial(n=10, p=0.3)

For each combination, plot histogram of 1000 sample means and overlay normal curve.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

def generate_sampling_dist(pop_func, pop_params, sample_sizes, n_samples=1000):
    """Generate sampling distributions for different sample sizes"""
    results = {}
    for n in sample_sizes:
        sample_means = []
        for _ in range(n_samples):
            sample = pop_func(**pop_params, size=n)
            sample_means.append(np.mean(sample))
        results[n] = np.array(sample_means)
    return results

# Define populations
populations = {
    'Uniform(0,1)': {
        'func': np.random.uniform,
        'params': {'low': 0, 'high': 1}
    },
    'Exponential(lambda=1)': {
        'func': np.random.exponential,
        'params': {'scale': 1}
    },
    'Binomial(n=10,p=0.3)': {
        'func': np.random.binomial,
        'params': {'n': 10, 'p': 0.3}
    }
}

sample_sizes = [5, 20, 50]

# Generate and plot
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row_idx, (pop_name, pop_def) in enumerate(populations.items()):
    sampling_dists = generate_sampling_dist(pop_def['func'], pop_def['params'], sample_sizes)
    
    for col_idx, n in enumerate(sample_sizes):
        means = sampling_dists[n]
        mu, sigma = np.mean(means), np.std(means)
        
        axes[row_idx, col_idx].hist(means, bins=30, density=True, alpha=0.7, edgecolor='black')
        
        x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
        axes[row_idx, col_idx].plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal approx')
        
        sk = skew(means)
        axes[row_idx, col_idx].text(0.05, 0.95, f'Skew: {sk:.2f}',
                                    transform=axes[row_idx, col_idx].transAxes,
                                    fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        
        axes[row_idx, col_idx].set_xlabel('Sample Mean')
        axes[row_idx, col_idx].set_ylabel('Density')
        axes[row_idx, col_idx].set_title(f'{pop_name}\nn={n}')
        axes[row_idx, col_idx].legend(fontsize=8)
        axes[row_idx, col_idx].grid(alpha=0.3)

plt.suptitle('Central Limit Theorem: Sampling Distributions Approach Normality', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

# Summary table
print("\nSkewness of Sampling Distributions (closer to 0 = more normal):")
print(f"{'Population':<25} {'n=5':>10} {'n=20':>10} {'n=50':>10}")
print("-" * 55)
for pop_name, pop_def in populations.items():
    sampling_dists = generate_sampling_dist(pop_def['func'], pop_def['params'], sample_sizes)
    row = [pop_name]
    for n in sample_sizes:
        sk = skew(sampling_dists[n])
        row.append(f"{sk:.3f}")
    print(f"{row[0]:<25} {row[1]:>10} {row[2]:>10} {row[3]:>10}")

print("\nInsight: CLT works faster for symmetric populations; skewed populations need larger n!")
```
</details>


---

# Exercise 9: Standard Error

## Task

Calculate standard deviation and standard error. Compare both.

<details>
<summary>Click to view solution</summary>

```python
std = state["Population"].std()
se = sem(state["Population"])

print(f"Standard Deviation: {std:,.2f}")
print(f"Standard Error: {se:,.2f}")
print(f"Sample Size: {len(state)}")
print(f"Theoretical SE (sigma/sqrt(n)): {std / np.sqrt(len(state)):,.2f}")
```
</details>

## Reflection

Questions:
1. Which value is larger?
2. Why?
3. What happens if sample size increases?



---

# Exercise 10: Standard Error in Practice

## Task

Using loan income data (or simulated lognormal data):
1. Take a sample of n=100
2. Compute theoretical SE = s/√n
3. Use bootstrap (R=2000) to estimate empirical SE
4. Compare theoretical vs. bootstrap SE
5. Repeat for n=400 and verify SE decreases by factor of ~2

<details>
<summary>Click to view solution</summary>

```python
# Load or simulate loan income data
try:
    income_data = pd.read_csv('../datasets/raw/loans_income.csv')['income'].dropna().values
except:
    np.random.seed(42)
    income_data = np.random.lognormal(mean=10, sigma=1.2, size=10000)

def compare_se_methods(data, sample_size, R=2000, random_state=42):
    """Compare theoretical and bootstrap standard error estimates"""
    np.random.seed(random_state)
    
    sample = np.random.choice(data, size=sample_size, replace=False)
    
    s = np.std(sample, ddof=1)
    se_theoretical = s / np.sqrt(sample_size)
    
    bootstrap_means = bootstrap_statistic(sample, np.mean, R=R, random_state=random_state)
    se_bootstrap = np.std(bootstrap_means)
    
    return {
        'n': sample_size,
        'sample_mean': np.mean(sample),
        'sample_sd': s,
        'se_theoretical': se_theoretical,
        'se_bootstrap': se_bootstrap,
        'ratio': se_bootstrap / se_theoretical
    }

# Compare for n=100 and n=400
results = []
for n in [100, 400]:
    result = compare_se_methods(income_data, sample_size=n)
    results.append(result)
    print(f"\nSample Size n={n}:")
    print(f"  Sample Mean: ${result['sample_mean']:,.0f}")
    print(f"  Sample SD: ${result['sample_sd']:,.0f}")
    print(f"  Theoretical SE: ${result['se_theoretical']:,.0f}")
    print(f"  Bootstrap SE: ${result['se_bootstrap']:,.0f}")
    print(f"  Ratio (Bootstrap/Theoretical): {result['ratio']:.2f}")

# Verify 1/sqrt(n) relationship
se1 = results[0]['se_bootstrap']
se2 = results[1]['se_bootstrap']
expected_ratio = np.sqrt(400/100)
actual_ratio = se1/se2

print(f"\nSE Scaling Verification:")
print(f"  SE(n=100) / SE(n=400) = {actual_ratio:.2f}")
print(f"  Expected (sqrt(400)/sqrt(100)) = {expected_ratio:.2f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.bar(['n=100', 'n=400'],
        [results[0]['se_bootstrap'], results[1]['se_bootstrap']],
        color=['skyblue', 'lightgreen'], edgecolor='black')
plt.ylabel('Standard Error of Mean ($)')
plt.title('Standard Error Decreases as 1/sqrt(n)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nML Relevance: Understanding SE helps size validation sets and interpret metric uncertainty!")
```
</details>


---

# Exercise 11: Bootstrap Sampling

## Task

Create 1000 bootstrap samples and calculate mean distribution.

<details>
<summary>Click to view solution</summary>

```python
bootstrap_means = []

for _ in range(1000):
    bootstrap_sample = state.sample(frac=1, replace=True)
    bootstrap_means.append(bootstrap_sample["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, bins=30, kde=True)
plt.title("Bootstrap Mean Distribution")
plt.xlabel("Mean Population")
plt.show()

print(f"Bootstrap Mean: {np.mean(bootstrap_means):,.2f}")
print(f"Bootstrap Std: {np.std(bootstrap_means):,.2f}")
print(f"Original Mean: {state['Population'].mean():,.2f}")
```
</details>

## Reflection

Questions:
1. Why sample with replacement?
2. Why is bootstrap powerful?
3. Where is bootstrap used in ML?

---



# Exercise 12: Bootstrap for Median

## Task

Using loan income data:
1. Take a sample of n=50
2. Compute sample median
3. Use bootstrap (R=5000) to estimate standard error and 95% confidence interval
4. Compare bootstrap CI to a naive "±2 SE" interval

<details>
<summary>Click to view solution</summary>

```python
# Load data
try:
    income_data = pd.read_csv('../datasets/raw/loans_income.csv')['income'].dropna().values
except:
    np.random.seed(42)
    income_data = np.random.lognormal(mean=10, sigma=1.2, size=10000)

# Take small sample
np.random.seed(42)
sample = np.random.choice(income_data, size=50, replace=False)

print(f"Sample Statistics (n={len(sample)}):")
print(f"  Mean: ${np.mean(sample):,.0f}")
print(f"  Median: ${np.median(sample):,.0f}")
print(f"  SD: ${np.std(sample, ddof=1):,.0f}")

# Bootstrap for median
bootstrap_medians = bootstrap_statistic(sample, np.median, R=5000, random_state=42)

# Bootstrap SE and CIs
se_bootstrap = np.std(bootstrap_medians)
ci_90 = np.percentile(bootstrap_medians, [5, 95])
ci_95 = np.percentile(bootstrap_medians, [2.5, 97.5])

# Naive "±2 SE" interval
naive_ci = [np.median(sample) - 2*se_bootstrap, np.median(sample) + 2*se_bootstrap]

print(f"\nBootstrap Results for Median:")
print(f"  Bootstrap SE: ${se_bootstrap:,.0f}")
print(f"  90% CI (percentile): [${ci_90[0]:,.0f}, ${ci_90[1]:,.0f}]")
print(f"  95% CI (percentile): [${ci_95[0]:,.0f}, ${ci_95[1]:,.0f}]")
print(f"  Naive ±2 SE CI: [${naive_ci[0]:,.0f}, ${naive_ci[1]:,.0f}]")

# Check normality of bootstrap distribution
skewness = skew(bootstrap_medians)
shapiro_stat, shapiro_p = shapiro(bootstrap_medians[:2000])  # Use subset for speed

print(f"\nBootstrap Distribution Diagnostics:")
print(f"  Skewness: {skewness:.3f}")
print(f"  Shapiro-Wilk p-value: {shapiro_p:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bootstrap_medians, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[0].axvline(np.median(sample), color='blue', linestyle='--', label='Sample median')
axes[0].axvline(ci_95[0], color='red', linestyle=':', label='95% CI bounds')
axes[0].axvline(ci_95[1], color='red', linestyle=':')
axes[0].axvline(naive_ci[0], color='orange', linestyle='-.', alpha=0.7, label='Naive ±2 SE')
axes[0].axvline(naive_ci[1], color='orange', linestyle='-.', alpha=0.7)
axes[0].set_xlabel('Bootstrap Median')
axes[0].set_ylabel('Density')
axes[0].set_title('Bootstrap Distribution of Median')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

stats.probplot(bootstrap_medians, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: Bootstrap Medians vs Normal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInsight: When bootstrap distribution is skewed, percentile CI ≠ naive ±2 SE!")
```
</details>


---

# Exercise 13: Bootstrap for Correlation Coefficient

## Task

Using `state.csv`:
1. Compute Pearson correlation between `Population` and `Murder.rate`
2. Use bootstrap (R=3000) to estimate standard error and 95% CI
3. Test if correlation is significantly different from 0
4. Repeat with Spearman correlation—do conclusions change?

<details>
<summary>Click to view solution</summary>

```python
# Prepare data
data = state[['Population', 'Murder.rate']].dropna().values

# Function to compute correlation from resampled data
def corr_statistic(data, method='pearson'):
    """Compute correlation from 2-column array"""
    if len(data) < 3:
        return np.nan
    df_temp = pd.DataFrame(data, columns=['Pop', 'Murder'])
    return df_temp['Pop'].corr(df_temp['Murder'], method=method)

# Bootstrap for Pearson correlation
np.random.seed(42)
pearson_obs = corr_statistic(data, method='pearson')
bootstrap_pearson = bootstrap_statistic(data, lambda x: corr_statistic(x, 'pearson'), R=3000, random_state=42)

# Bootstrap for Spearman correlation
spearman_obs = corr_statistic(data, method='spearman')
bootstrap_spearman = bootstrap_statistic(data, lambda x: corr_statistic(x, 'spearman'), R=3000, random_state=42)

# Results for Pearson
pearson_se = np.std(bootstrap_pearson)
pearson_ci = np.percentile(bootstrap_pearson, [2.5, 97.5])
pearson_sig = "Yes" if (pearson_ci[0] > 0 or pearson_ci[1] < 0) else "No"

# Results for Spearman
spearman_se = np.std(bootstrap_spearman)
spearman_ci = np.percentile(bootstrap_spearman, [2.5, 97.5])
spearman_sig = "Yes" if (spearman_ci[0] > 0 or spearman_ci[1] < 0) else "No"

print("Correlation: Population vs Murder Rate")
print(f"\n{'Method':<12} {'Observed':>10} {'SE':>8} {'95% CI':>20} {'Sig? (≠0)':>10}")
print("-" * 65)
print(f"{'Pearson':<12} {pearson_obs:>10.3f} {pearson_se:>8.3f} [{pearson_ci[0]:.3f}, {pearson_ci[1]:.3f}] {pearson_sig:>10}")
print(f"{'Spearman':<12} {spearman_obs:>10.3f} {spearman_se:>8.3f} [{spearman_ci[0]:.3f}, {spearman_ci[1]:.3f}] {spearman_sig:>10}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bootstrap_pearson, bins=40, density=True, alpha=0.7, edgecolor='black', color='skyblue')
axes[0].axvline(pearson_obs, color='blue', linestyle='--', label=f'Observed: {pearson_obs:.3f}')
axes[0].axvline(0, color='red', linestyle=':', label='Null (r=0)')
axes[0].axvline(pearson_ci[0], color='green', linestyle='-.', label='95% CI')
axes[0].axvline(pearson_ci[1], color='green', linestyle='-.')
axes[0].set_xlabel('Bootstrap Correlation (Pearson)')
axes[0].set_ylabel('Density')
axes[0].set_title('Bootstrap Distribution: Pearson Correlation')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].hist(bootstrap_spearman, bins=40, density=True, alpha=0.7, edgecolor='black', color='lightgreen')
axes[1].axvline(spearman_obs, color='darkgreen', linestyle='--', label=f'Observed: {spearman_obs:.3f}')
axes[1].axvline(0, color='red', linestyle=':', label='Null (ρ=0)')
axes[1].axvline(spearman_ci[0], color='orange', linestyle='-.', label='95% CI')
axes[1].axvline(spearman_ci[1], color='orange', linestyle='-.')
axes[1].set_xlabel('Bootstrap Correlation (Spearman)')
axes[1].set_ylabel('Density')
axes[1].set_title('Bootstrap Distribution: Spearman Correlation')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
```
</details>



---

# Exercise 14: Confidence Intervals

## Task

Use bootstrap results to calculate 95% confidence interval.

<details>
<summary>Click to view solution</summary>

```python
confidence_interval = np.percentile(bootstrap_means, [2.5, 97.5])

print("95% Confidence Interval")
print(f"Lower Bound: {confidence_interval[0]:,.2f}")
print(f"Upper Bound: {confidence_interval[1]:,.2f}")

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, bins=30, kde=True)
plt.axvline(confidence_interval[0], linestyle="--", label="Lower Bound")
plt.axvline(confidence_interval[1], linestyle="--", label="Upper Bound")
plt.legend()
plt.title("95% Confidence Interval")
plt.show()
```
</details>

## Reflection

Questions:
1. Why are intervals useful?
2. What happens if uncertainty increases?
3. Why should businesses care about confidence intervals?

---

# Exercise 15: Comparing CI Methods for Skewed Data

## Task

Using right-skewed loan income data:
1. Take a sample of n=200
2. Compute 95% CIs for the mean using formula-based (t-distribution) and bootstrap percentile
3. Which interval is widest? Which is most symmetric?
4. Simulate 100 samples and compute coverage

<details>
<summary>Click to view solution</summary>

```python
# Load or simulate data
try:
    income_data = pd.read_csv('../datasets/raw/loans_income.csv')['income'].dropna().values
except:
    np.random.seed(42)
    income_data = np.random.lognormal(mean=10, sigma=1.2, size=50000)

true_mean = np.mean(income_data)
print(f"True Population Mean: ${true_mean:,.0f}")

def ci_t_distribution(sample, confidence=0.95):
    """Formula-based CI using t-distribution"""
    n = len(sample)
    mean = np.mean(sample)
    se = np.std(sample, ddof=1) / np.sqrt(n)
    t_crit = t.ppf((1 + confidence) / 2, df=n-1)
    return mean - t_crit * se, mean + t_crit * se

def ci_bootstrap_percentile(sample, R=2000, confidence=0.95, random_state=None):
    """Bootstrap percentile CI"""
    if random_state is not None:
        np.random.seed(random_state)
    bootstrap_means = bootstrap_statistic(sample, np.mean, R=R)
    alpha = 1 - confidence
    return np.percentile(bootstrap_means, [100*alpha/2, 100*(1-alpha/2)])

# Take one sample
np.random.seed(42)
sample = np.random.choice(income_data, size=200, replace=False)

# Compute CIs
ci_t = ci_t_distribution(sample)
ci_boot = ci_bootstrap_percentile(sample, R=3000, random_state=42)

print(f"\n95% Confidence Intervals for Mean (n=200):")
print(f"{'Method':<25} {'Lower':>12} {'Upper':>12} {'Width':>10}")
print("-" * 60)

sample_mean = np.mean(sample)
for name, ci in [('t-distribution', ci_t), ('Bootstrap Percentile', ci_boot)]:
    lower, upper = ci
    width = upper - lower
    asymmetry = abs((upper - sample_mean) - (sample_mean - lower)) / width
    symmetric = "Yes" if asymmetry < 0.05 else f"No (asym={asymmetry:.2f})"
    print(f"{name:<25} ${lower:>10,.0f} ${upper:>10,.0f} ${width:>8,.0f}  {symmetric}")

# Coverage simulation
print(f"\nCoverage Simulation (100 samples):")
n_sim = 100
coverage_t = 0
coverage_boot = 0

for i in range(n_sim):
    sim_sample = np.random.choice(income_data, size=200, replace=False)
    
    ci_t_sim = ci_t_distribution(sim_sample)
    if ci_t_sim[0] <= true_mean <= ci_t_sim[1]:
        coverage_t += 1
    
    ci_boot_sim = ci_bootstrap_percentile(sim_sample, R=1000, random_state=i)
    if ci_boot_sim[0] <= true_mean <= ci_boot_sim[1]:
        coverage_boot += 1

print(f"  t-distribution CI coverage: {coverage_t}/{n_sim} = {coverage_t/n_sim:.2%}")
print(f"  Bootstrap percentile coverage: {coverage_boot}/{n_sim} = {coverage_boot/n_sim:.2%}")
print(f"  Expected (95% CI): ~95%")

print("\nInsight: For skewed data, bootstrap CIs may have better coverage!")
```
</details>


---

# Exercise 16: CI for Proportion (Binomial Case)

## Task

Simulate an A/B test with binary outcome:
- True conversion rate: 12%
- Sample size: n=500

Compute 95% CIs using Wald interval, Wilson score interval, and bootstrap percentile. Compare widths and coverage.

<details>
<summary>Click to view solution</summary>

```python
from statsmodels.stats.proportion import proportion_confint

np.random.seed(42)

# True parameters
p_true = 0.12
n = 500

def ci_wald(conversions, n, confidence=0.95):
    """Wald interval (normal approximation)"""
    p_hat = conversions / n
    z = norm.ppf((1 + confidence) / 2)
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat - z * se, p_hat + z * se

def ci_bootstrap_prop(conversions, n, R=2000, confidence=0.95, random_state=None):
    """Bootstrap CI for proportion"""
    if random_state is not None:
        np.random.seed(random_state)
    
    data = np.concatenate([np.ones(conversions), np.zeros(n - conversions)])
    
    bootstrap_props = []
    for _ in range(R):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_props.append(np.mean(resample))
    
    alpha = 1 - confidence
    return np.percentile(bootstrap_props, [100*alpha/2, 100*(1-alpha/2)])

# Generate one sample
conversions = np.random.binomial(n, p_true)
p_hat = conversions / n

print(f"One Sample: {conversions}/{n} conversions (p_hat = {p_hat:.3f})")

# Compute CIs
ci_w = ci_wald(conversions, n)
ci_wilson = proportion_confint(conversions, n, alpha=0.05, method='wilson')
ci_boot = ci_bootstrap_prop(conversions, n, R=3000, random_state=42)

print(f"\n95% Confidence Intervals for Proportion:")
print(f"{'Method':<20} {'Lower':>10} {'Upper':>10} {'Width':>8}")
print("-" * 50)
for name, ci in [('Wald', ci_w), ('Wilson', ci_wilson), ('Bootstrap', ci_boot)]:
    lower, upper = ci
    print(f"{name:<20} {lower:>10.3f} {upper:>10.3f} {upper-lower:>8.3f}")

# Coverage simulation
print(f"\nCoverage Simulation (100 samples, true p={p_true}):")
n_sim = 100
coverage = {'Wald': 0, 'Wilson': 0, 'Bootstrap': 0}

for i in range(n_sim):
    conv_sim = np.random.binomial(n, p_true)
    
    ci_w_sim = ci_wald(conv_sim, n)
    if ci_w_sim[0] <= p_true <= ci_w_sim[1]:
        coverage['Wald'] += 1
    
    ci_wil_sim = proportion_confint(conv_sim, n, alpha=0.05, method='wilson')
    if ci_wil_sim[0] <= p_true <= ci_wil_sim[1]:
        coverage['Wilson'] += 1
    
    ci_b_sim = ci_bootstrap_prop(conv_sim, n, R=1000, random_state=i)
    if ci_b_sim[0] <= p_true <= ci_b_sim[1]:
        coverage['Bootstrap'] += 1

for method in ['Wald', 'Wilson', 'Bootstrap']:
    cov_rate = coverage[method] / n_sim
    print(f"  {method:<20}: {cov_rate:>10.2%}")
print(f"  Expected (95% CI): ~95%")

print("\nInsight: Wilson interval often has better coverage than Wald for proportions!")
```
</details>


---

# Exercise 17: Regression to the Mean

## Task

Generate synthetic performance data:
- True skill ~ N(0, 1) for 1000 individuals
- Observed performance = skill + noise, where noise ~ N(0, 2)
- Select top 10% by observed performance in "Period 1"
- Compare their Period 2 performance to their Period 1 performance

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Generate data
n = 1000
skill = np.random.normal(0, 1, n)
noise_p1 = np.random.normal(0, 2, n)
noise_p2 = np.random.normal(0, 2, n)

performance_p1 = skill + noise_p1
performance_p2 = skill + noise_p2

# Select top 10% in Period 1
threshold_p1 = np.percentile(performance_p1, 90)
top_10_p1 = performance_p1 >= threshold_p1

# Analyze selected group
selected_p1 = performance_p1[top_10_p1]
selected_p2 = performance_p2[top_10_p1]

print(f"Top 10% Performers in Period 1 (n={np.sum(top_10_p1)}):")
print(f"  Period 1 Mean Performance: {np.mean(selected_p1):.2f}")
print(f"  Period 2 Mean Performance: {np.mean(selected_p2):.2f}")
print(f"  Regression toward mean: {np.mean(selected_p1) - np.mean(selected_p2):.2f}")

# Stability: % remaining in top 10% in Period 2
threshold_p2 = np.percentile(performance_p2, 90)
still_top_10 = performance_p2[top_10_p1] >= threshold_p2
stability_rate = np.mean(still_top_10)
print(f"  Stability: {stability_rate:.2%} remain in top 10% in Period 2")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(selected_p1, selected_p2, alpha=0.6, edgecolors='black')
axes[0].plot([performance_p1.min(), performance_p1.max()],
             [performance_p1.min(), performance_p1.max()],
             'r--', label='Perfect correlation')
axes[0].set_xlabel('Period 1 Performance')
axes[0].set_ylabel('Period 2 Performance')
axes[0].set_title('Top 10% Period 1: P1 vs P2')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(selected_p1, bins=20, alpha=0.5, label='Period 1', density=True)
axes[1].hist(selected_p2, bins=20, alpha=0.5, label='Period 2', density=True)
axes[1].axvline(np.mean(selected_p1), color='blue', linestyle='--', label='P1 Mean')
axes[1].axvline(np.mean(selected_p2), color='green', linestyle='--', label='P2 Mean')
axes[1].set_xlabel('Performance')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution Shift: Regression to Mean')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
```
</details>


---

# Exercise 18: Avoiding Regression Fallacy in A/B Testing

## Task

Simulate an A/B test with no true effect (both groups 10% conversion rate, n=1000 per group). Run 100 times:
1. Count false positive rate
2. Calculate average observed lift for "winning" tests
3. Simulate retest of "winning" variants—do they replicate?

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

def run_ab_test(n_per_group=1000, p_control=0.10, p_treatment=0.10):
    """Simulate one A/B test, return p-value and observed lift"""
    control_conversions = np.random.binomial(n_per_group, p_control)
    treatment_conversions = np.random.binomial(n_per_group, p_treatment)
    
    control_rate = control_conversions / n_per_group
    treatment_rate = treatment_conversions / n_per_group
    observed_lift = treatment_rate - control_rate
    
    count = np.array([treatment_conversions, control_conversions])
    nobs = np.array([n_per_group, n_per_group])
    stat, p_value = proportions_ztest(count, nobs, alternative='larger')
    
    return p_value, observed_lift, control_rate, treatment_rate

# Run 100 simulations under null hypothesis
results = []
for i in range(100):
    p_val, lift, ctrl_rate, treat_rate = run_ab_test()
    results.append({
        'simulation': i,
        'p_value': p_val,
        'observed_lift': lift,
        'significant': p_val < 0.05
    })

results_df = pd.DataFrame(results)

# False positive rate
false_positives = results_df['significant'].sum()
print(f"False Positive Rate (alpha=0.05, null true):")
print(f"  Significant results by chance: {false_positives}/100 = {false_positives}%")
print(f"  Expected: ~5/100 = 5%")

# Average lift for "winning" tests
winners = results_df[results_df['significant']]
if len(winners) > 0:
    avg_lift_winners = winners['observed_lift'].mean()
    print(f"\nAverage Observed Lift for 'Winning' Tests:")
    print(f"  Mean lift: {avg_lift_winners:.2%} (true lift = 0%)")
    print(f"  Overestimate due to regression to mean!")

# Retest simulation
if len(winners) > 0:
    print(f"\nRetest Simulation for 'Winning' Variants:")
    retest_results = []
    for _, row in winners.iterrows():
        p_val_retest, lift_retest, _, _ = run_ab_test()
        retest_results.append({
            'original_lift': row['observed_lift'],
            'retest_lift': lift_retest,
            'retest_significant': p_val_retest < 0.05
        })
    
    retest_df = pd.DataFrame(retest_results)
    replication_rate = retest_df['retest_significant'].mean()
    avg_retest_lift = retest_df['retest_lift'].mean()
    
    print(f"  Replication rate (significant in retest): {replication_rate:.2%}")
    print(f"  Average lift in retest: {avg_retest_lift:.2%}")
    print(f"  Most 'wins' don't replicate due to regression to mean!")

    # Visualization
    plt.figure(figsize=(8, 6))
    plt.scatter(retest_df['original_lift'], retest_df['retest_lift'], alpha=0.7, edgecolors='black')
    plt.axhline(y=0, color='green', linestyle='--', label='True Lift (0%)')
    plt.axvline(x=0, color='green', linestyle='--')
    plt.plot([-0.05, 0.05], [-0.05, 0.05], 'r:', label='Perfect replication')
    plt.xlabel('Original Observed Lift')
    plt.ylabel('Retest Observed Lift')
    plt.title('Retest Performance of "Winning" A/B Tests')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\nML Relevance: Always validate model improvements on fresh holdout data to avoid regression fallacy!")
```
</details>



---

# Exercise 19: Normal Distribution

## Task

Generate a synthetic normal distribution. Observe symmetry, centre, spread.

<details>
<summary>Click to view solution</summary>

```python
normal_data = np.random.normal(loc=50, scale=10, size=10000)

plt.figure(figsize=(10, 6))
sns.histplot(normal_data, kde=True)
plt.title("Normal Distribution")
plt.show()

print(f"Mean: {np.mean(normal_data):.2f}")
print(f"Median: {np.median(normal_data):.2f}")
print(f"Standard Deviation: {np.std(normal_data):.2f}")
print(f"Skewness: {skew(normal_data):.3f}")
```
</details>

## Questions

1. Why does the distribution look symmetric?
2. Why are mean and median similar?
3. Why do many methods assume normality?



---

# Exercise 20: Long-tail Distribution

## Task

Create a skewed distribution. Compare it with normal distribution.

<details>
<summary>Click to view solution</summary>

```python
skewed_data = np.random.exponential(scale=2, size=10000)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(normal_data, kde=True, ax=ax[0])
ax[0].set_title("Normal Distribution")

sns.histplot(skewed_data, bins=40, kde=True, ax=ax[1])
ax[1].set_title("Long-tail Distribution")

plt.show()

print(f"Normal: Mean={np.mean(normal_data):.2f}, Median={np.median(normal_data):.2f}")
print(f"Skewed: Mean={np.mean(skewed_data):.2f}, Median={np.median(skewed_data):.2f}")
```
</details>

## Reflection

Questions:
1. Why is mean larger than median?
2. Which metric is safer?
3. Why are real-world datasets often skewed?


---

# Exercise 21: QQ-Plot Diagnostic Challenge

## Task

Analyze 4 unknown datasets (Normal, Exponential, Uniform, t-distribution). For each:
1. Create histogram + KDE
2. Create QQ-plot against normal distribution
3. Compute skewness and kurtosis
4. Hypothesize the generating distribution
5. Verify with goodness-of-fit test

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Generate 4 datasets with different distributions
datasets = {
    'A': norm.rvs(loc=50, scale=10, size=500),
    'B': expon.rvs(scale=5, size=500),
    'C': uniform.rvs(loc=20, scale=40, size=500),
    'D': t.rvs(df=3, size=500)
}

def analyze_distribution(data, name):
    print(f"\nDataset {name}:")
    print(f"  Mean: {np.mean(data):.2f}, SD: {np.std(data):.2f}")
    print(f"  Skewness: {skew(data):.3f}, Kurtosis: {kurtosis(data):.3f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    sns.histplot(data, kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'{name}: Histogram + KDE')
    axes[0].grid(alpha=0.3)
    
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].set_title(f'{name}: QQ-Plot vs Normal')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    sk = skew(data)
    kt = kurtosis(data)
    
    if abs(sk) < 0.5 and abs(kt) < 0.5:
        hypothesis = "Normal"
    elif sk > 1 and kt > 2:
        hypothesis = "Exponential"
    elif abs(sk) < 0.3 and kt < -1:
        hypothesis = "Uniform"
    elif abs(sk) < 0.5 and kt > 2:
        hypothesis = "t-distribution"
    else:
        hypothesis = "Unknown"
    
    print(f"  Hypothesis: {hypothesis}")
    return hypothesis

# Analyze each dataset
hypotheses = {}
for name, data in datasets.items():
    hypotheses[name] = analyze_distribution(data, name)

# Verification
print(f"\nGoodness-of-Fit Verification:")
print(f"{'Dataset':<10} {'Hypothesis':<15} {'p-value':>10} {'Conclusion':>12}")
print("-" * 50)

for name, data in datasets.items():
    hypo = hypotheses[name]
    
    if hypo == "Normal":
        mu, sigma = np.mean(data), np.std(data)
        stat, p = kstest(data, 'norm', args=(mu, sigma))
    elif hypo == "Exponential":
        scale = np.mean(data)
        stat, p = kstest(data, 'expon', args=(0, scale))
    elif hypo == "Uniform":
        loc, scale = np.min(data), np.max(data) - np.min(data)
        stat, p = kstest(data, 'uniform', args=(loc, scale))
    elif hypo == "t-distribution":
        stat, p = kstest(data, 't', args=(3,))
    else:
        stat, p = np.nan, np.nan
    
    conclusion = "Fit" if p > 0.05 else "Poor fit"
    print(f"{name:<10} {hypo:<15} {p:>10.4f} {conclusion:>12}")
```
</details>



---

# Exercise 22: Choosing the Right Distribution for Modeling

## Task

Fit three candidate distributions to right-skewed session duration data: Exponential, Gamma, Lognormal. Compare fits using AIC and select best.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Simulate realistic session duration data (right-skewed)
short_sessions = expon.rvs(scale=5, size=800)
long_sessions = gamma.rvs(a=2, scale=15, size=200)
session_data = np.concatenate([short_sessions, long_sessions])
session_data = session_data[session_data > 0]

print(f"Session Duration Data (n={len(session_data)}):")
print(f"  Mean: {np.mean(session_data):.2f} min, Median: {np.median(session_data):.2f} min")
print(f"  Skewness: {skew(session_data):.3f}")

# Fit candidate distributions
fits = {}

exp_params = expon.fit(session_data)
fits['Exponential'] = {'dist': expon(*exp_params), 'k': 1}

gamma_params = gamma.fit(session_data, floc=0)
fits['Gamma'] = {'dist': gamma(*gamma_params), 'k': 2}

lognorm_params = lognorm.fit(session_data, floc=0)
fits['Lognormal'] = {'dist': lognorm(*lognorm_params), 'k': 2}

# Compute AIC
print(f"\nModel Comparison:")
print(f"{'Distribution':<15} {'Log-Likelihood':>18} {'AIC':>12} {'95th Percentile':>18}")
print("-" * 70)

results = []
for name, fit in fits.items():
    dist = fit['dist']
    k = fit['k']
    
    log_likelihood = np.sum(dist.logpdf(session_data))
    aic = 2*k - 2*log_likelihood
    p95 = dist.ppf(0.95)
    
    results.append({'name': name, 'loglik': log_likelihood, 'aic': aic, 'p95': p95})
    print(f"{name:<15} {log_likelihood:>18.2f} {aic:>12.2f} {p95:>18.2f} min")

# Select best by AIC
best = min(results, key=lambda x: x['aic'])
print(f"\nBest model by AIC: {best['name']}")
print(f"  95th percentile session duration: {best['p95']:.2f} minutes")

# Visualization
plt.figure(figsize=(10, 6))
plt.hist(session_data, bins=50, density=True, alpha=0.5, label='Data', edgecolor='black')

x = np.linspace(0, np.percentile(session_data, 99), 200)
colors = ['red', 'green', 'blue']
for idx, (name, fit) in enumerate(fits.items()):
    pdf = fit['dist'].pdf(x)
    plt.plot(x, pdf, color=colors[idx], linewidth=2, label=f"{name} fit")

plt.xlabel('Session Duration (minutes)')
plt.ylabel('Density')
plt.title('Fitting Distributions to Session Duration Data')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nML Relevance: Choosing the right distribution improves simulation, anomaly detection, and forecasting!")
```
</details>



---

# Exercise 23: End-to-End Sampling Analysis

## Task

Analyze user engagement for a mobile app with 365 days of daily active users (DAU) data:
1. Simulate DAU data (include seasonality + noise)
2. Take a sample of one random week
3. Use bootstrap to estimate 95% CI for mean DAU
4. Compare to using all 365 days
5. Write summary for a product manager

<details>
<summary>Click to view solution</summary>

```python
from statsmodels.tsa.stattools import adfuller

np.random.seed(42)

# Simulate DAU data with seasonality + trend + noise
days = np.arange(365)
base = 50000
trend = 50 * days / 365
seasonal = 5000 * np.sin(2 * np.pi * days / 7)
noise = np.random.normal(0, 3000, 365)
dau = np.maximum(base + trend + seasonal + noise, 0)

print(f"DAU Data Summary:")
print(f"  Range: {dau.min():,.0f} to {dau.max():,.0f}")
print(f"  Mean: {np.mean(dau):,.0f}, SD: {np.std(dau):,.0f}")

# Visualize
plt.figure(figsize=(12, 4))
plt.plot(days, dau, linewidth=1)
plt.xlabel('Day')
plt.ylabel('Daily Active Users')
plt.title('Simulated DAU Over 365 Days')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Take a "sample": one random week
np.random.seed(42)
week_start = np.random.randint(0, 365-7)
sample_days = np.arange(week_start, week_start + 7)
sample_dau = dau[sample_days]

print(f"\nSample: Days {week_start}-{week_start+6}")
print(f"  Sample Mean DAU: {np.mean(sample_dau):,.0f}")

# Bootstrap CI for sample
bootstrap_sample_means = bootstrap_statistic(sample_dau, np.mean, R=5000, random_state=42)
sample_ci = np.percentile(bootstrap_sample_means, [2.5, 97.5])

# Full data bootstrap
bootstrap_full_means = bootstrap_statistic(dau, np.mean, R=2000, random_state=42)
full_ci = np.percentile(bootstrap_full_means, [2.5, 97.5])

print(f"\nConfidence Intervals for Mean DAU:")
print(f"{'Data':<15} {'Mean':>12} {'95% CI':>25} {'Width':>10}")
print("-" * 65)
print(f"{'Sample (n=7)':<15} {np.mean(sample_dau):>10,.0f} [{sample_ci[0]:>8,.0f}, {sample_ci[1]:>8,.0f}] {sample_ci[1]-sample_ci[0]:>8,.0f}")
print(f"{'Full (n=365)':<15} {np.mean(dau):>10,.0f} [{full_ci[0]:>8,.0f}, {full_ci[1]:>8,.0f}] {full_ci[1]-full_ci[0]:>8,.0f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bootstrap_sample_means, bins=40, density=True, alpha=0.7, edgecolor='black', color='skyblue')
axes[0].axvline(np.mean(sample_dau), color='blue', linestyle='--')
axes[0].axvline(sample_ci[0], color='red', linestyle=':')
axes[0].axvline(sample_ci[1], color='red', linestyle=':')
axes[0].set_xlabel('Bootstrap Mean DAU')
axes[0].set_ylabel('Density')
axes[0].set_title('Sample (n=7): Bootstrap Distribution')
axes[0].grid(alpha=0.3)

axes[1].hist(bootstrap_full_means, bins=40, density=True, alpha=0.7, edgecolor='black', color='lightgreen')
axes[1].axvline(np.mean(dau), color='darkgreen', linestyle='--')
axes[1].axvline(full_ci[0], color='orange', linestyle=':')
axes[1].axvline(full_ci[1], color='orange', linestyle=':')
axes[1].set_xlabel('Bootstrap Mean DAU')
axes[1].set_ylabel('Density')
axes[1].set_title('Full Data (n=365): Bootstrap Distribution')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Product manager summary
print(f"\nSummary for Product Manager:")
print(f"  1. Based on our data, typical daily active users is approximately {np.mean(dau):,.0f}.")
print(f"  2. With only one week of data, uncertainty is high (±{int((sample_ci[1]-sample_ci[0])/2):,.0f} users).")
print(f"  3. Using the full year reduces uncertainty to ±{int((full_ci[1]-full_ci[0])/2):,.0f} users, enabling more confident decisions.")
```
</details>


---

# Mini Experiment: Sample Size Comparison

## Question

What happens if sample size changes? Compare n=5 and n=50.

<details>
<summary>Click to view solution</summary>

```python
sample_5 = []
sample_50 = []

for _ in range(1000):
    sample_5.append(state.sample(5, replace=True)["Population"].mean())
    sample_50.append(state.sample(50, replace=True)["Population"].mean())

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(sample_5, kde=True, ax=ax[0])
ax[0].set_title("Sample Size = 5")

sns.histplot(sample_50, kde=True, ax=ax[1])
ax[1].set_title("Sample Size = 50")

plt.show()
```
</details>

## Reflection

Questions:
1. Which distribution is tighter?
2. Why do larger samples reduce uncertainty?
3. Why is more representative data valuable?

---



# ML Relevance Exercise

## Question

Why does Chapter 2 matter in Machine Learning?

Think about:
- Train/test split
- Sampling bias
- Bootstrap
- Generalisation
- Confidence intervals
- Uncertainty

---

# Interview Questions

1. **Difference between population and sample?**
2. **What is sampling bias?**
3. **Explain Central Limit Theorem.**
4. **Difference between standard deviation and standard error?**
5. **What is bootstrap?**
6. **What is a confidence interval?**
7. **Why do larger samples reduce uncertainty?**
8. **Why are business datasets often skewed?**
9. **What is regression to the mean?**
10. **When would you use t-distribution vs normal distribution?**
11. **Why is the bootstrap so valuable compared to classical formula-based inference?**

---

# Reflection Questions

1. **When might a 90% confidence interval be preferable to a 95% interval in a business context?**
2. **How does understanding sampling distributions help you design better train/test splits?**
3. **A colleague says: "The bootstrap gave me a narrow confidence interval, so my estimate must be precise." What's wrong with this reasoning?**

---

# Challenge Problems

1. Create a simulation showing sample size vs standard error
2. Build biased vs unbiased sample comparison
3. Create 1000 bootstrap confidence intervals
4. Compare normal vs skewed distributions and explain implications for machine learning
5. Demonstrate regression to the mean with a real-world scenario

---

# Progress Checklist

- [ ] Practised population vs sample
- [ ] Explored random sampling
- [ ] Understood sampling bias and self-selection bias
- [ ] Simulated Literary Digest poll scenario
- [ ] Visualised sampling variability
- [ ] Understood Central Limit Theorem
- [ ] Visualized CLT with different population shapes
- [ ] Calculated standard error
- [ ] Compared theoretical vs bootstrap SE
- [ ] Practised bootstrap for mean, median, and correlation
- [ ] Built confidence intervals using multiple methods
- [ ] Compared CI methods for skewed data
- [ ] Computed CIs for proportions (Wald, Wilson, bootstrap)
- [ ] Simulated regression to the mean
- [ ] Tested regression fallacy in A/B testing
- [ ] Compared normal and long-tail distributions
- [ ] Performed QQ-plot diagnostics
- [ ] Fitted and compared distributions
- [ ] Completed end-to-end sampling analysis
- [ ] Finished mini experiment
- [ ] Answered interview questions
- [ ] Ready for Chapter 3


---
**Pro Tips**:  
> - When to use bootstrap vs. formula CIs  
> - CLT rules of thumb (n > 30? depends on skew!)  
> - Common distribution use cases  
> This becomes invaluable during modeling interviews or production debugging.
